In [0]:
%sql
CREATE VOLUME IF NOT EXISTS test_catalog.default.my_volume

In [0]:
catalog = "test_catalog"
schema = "default"
volume = "my_volume"
download_url = "https://health.data.ny.gov/api/views/jxy9-yhdk/rows.csv"
file_name = "baby_names.csv"
table_name = "baby_names"
path_volume = "/Volumes/" + catalog + "/" + schema + "/" + volume
path_table = catalog + "." + schema
print(path_table) # Show the complete path
print(path_volume) # Show the complete path

In [0]:
df = spark.read.csv(f"{path_volume}/{file_name}",
  header=True,
  inferSchema=True,
  sep=",")

display(df)

In [0]:
# Import functions
from pyspark.sql.functions import col, current_timestamp

# Define variables used in code below
catalog = "test_catalog"
schema = "default"
file_path = "/databricks-datasets/structured-streaming/events"
username = spark.sql("SELECT regexp_replace(session_user(), '[^a-zA-Z0-9]', '_')").first()[0]
table_name = f"{catalog}.{schema}.{username}_etl_quickstart"
checkpoint_path = f"/Volumes/{catalog}/{schema}/my_volume/_checkpoint/etl_quickstart"

# Clear out data from previous demo execution
spark.sql(f"DROP TABLE IF EXISTS {table_name}")
dbutils.fs.rm(checkpoint_path, True)

# Configure Auto Loader to ingest JSON data to a Delta table
(spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.schemaLocation", checkpoint_path)
  .load(file_path)
  .select("*", col("_metadata.file_path").alias("source_file"), current_timestamp().alias("processing_time"))
  .writeStream
  .option("checkpointLocation", checkpoint_path)
  .trigger(availableNow=True)
  .toTable(table_name))

In [0]:
df = spark.read.table(table_name)
display(df)

In [0]:
# file_path = f"/databricks-datasets/songs/data-001/"
# display(dbutils.fs.ls(file_path))

In [0]:
%sql
SELECT * FROM (
SELECT '00_default' AS Schema, 'source_clicks' AS TableName, COUNT(1) AS RowCount FROM test_catalog.default.source_clicks UNION ALL
SELECT '00_default' AS Schema, 'source_users' AS TableName, COUNT(1) AS RowCount FROM test_catalog.default.source_users UNION ALL
SELECT '01_bronze' AS Schema, 'clicks_bronze' AS TableName, COUNT(1) AS RowCount FROM test_catalog.bronze.clicks_bronze UNION ALL
SELECT '01_bronze' AS Schema, 'users_bronze' AS TableName, COUNT(1) AS RowCount FROM test_catalog.bronze.users_bronze UNION ALL
SELECT '02_silver' AS Schema, 'clicks_silver' AS TableName, COUNT(1) AS RowCount FROM test_catalog.silver.clicks_silver UNION ALL
SELECT '02_silver' AS Schema, 'users_silver' AS TableName, COUNT(1) AS RowCount FROM test_catalog.silver.users_silver UNION ALL
SELECT '02_silver' AS Schema, 'user_clicks_silver' AS TableName, COUNT(1) AS RowCount FROM test_catalog.silver.user_clicks_silver UNION ALL
SELECT '03_gold' AS Schema, 'user_behavior_gold' AS TableName, COUNT(1) AS RowCount FROM test_catalog.gold.user_behavior_gold UNION ALL
SELECT '03_gold' AS Schema, 'marketing_insights_gold' AS TableName, COUNT(1) AS RowCount FROM test_catalog.gold.marketing_insights_gold 
) A ORDER BY Schema



In [0]:
%sql
SHOW TBLPROPERTIES test_catalog.default.source_clicks